# 06 — Equal risk contribution: does equal risk remove the UUP corner?

`EqualRiskContribution` (in `maplab.models`): the long-only portfolio in which every asset carries the same share of variance, scored by the rolling backtest against EW and 60/40. It is risk-based: only Σ enters, no forecast of μ. Hypotheses are pre-registered in §2; cross-model tests are in notebooks 08–09. Run labels: (S) sample Σ, (LW) Ledoit–Wolf; IV is inverse volatility.

## §1 Risk parity and equal risk contribution

Risk parity allocates by risk instead of capital. In the equal-risk-contribution
(ERC) form every asset carries the same share of portfolio variance. ERC is the
minimum-variance portfolio subject to a floor on the sum of log weights; long-only,
that problem is strictly convex with a unique solution, which the class finds by
cyclical coordinate descent. The portfolio here is fully invested and unlevered.

Notebooks 04–05 found a UUP corner in GMV and MDP driven by correlation. ERC
fixes every asset's variance share at 1/N; §5 and §7 ask what that does to
UUP's capital weight, and §6 whether it changes returns beyond β.

## Setup

Prices, log returns for estimation, simple returns for the backtest, BIL as $r_f$, and the panel the classes read.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import maplab as ml
from maplab import (
    Panel, apply_style, GMV, MaxSharpe, EqualWeight, FixedWeight, MostDiversified,
    EqualRiskContribution, backtest, summary,
)
from maplab.models import _min_variance_long_only, _mdp_long_only, _erc_long_only, _tangency_long_only

from helpers import nb06 as h
from helpers import common as cm

In [ ]:
apply_style()

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"

In plain words: **H1** ERC holds less UUP than both GMV and MDP; **H2** ERC spreads its capital more than both (higher effective N); **H3** once market β is removed, ERC earns the same as GMV and MDP; **H4** ERC trails EW, mainly because it carries less market β; **H5** neither the Ledoit–Wolf estimator nor plain inverse volatility makes a detectable difference. The terms are defined in §3; §6 and §7 run the tests.

## §2 Pre-registered hypotheses

Registered before any backtest below was run; the frozen text is in
`registrations/nb06.toml`. ERC(S) is registered; ERC(LW) is the sensitivity and
inverse volatility (IV) the reference. The H1/H2 directions were fixed by a
read-only snapshot at 2014-12-31, 2017-12-31 and 2022-12-31; 2022-12-31 is
in-sample, so H1 and H2 must also hold on more than half of the 40 test
rebalances. H3 is on Δα because ERC runs at higher volatility than GMV on the
same Σ, so a raw return gap is expected. Paired tests use the daily return
difference, $t = \bar d / s_d \cdot \sqrt{n}$, plain OLS; a pass on the full
window only is reported as "not robust", never "supported".

| | Hypothesis | Counts if |
|---|---|---|
| H1 | UUP weight ERC < GMV and ERC < MDP | at 2022-12-31, on more than 50% of the 208 rebalances and more than 50% of the 40 test rebalances |
| H2 | effN(ERC) ≥ max(effN GMV, effN MDP) | on more than 50% of the 208 rebalances and more than 50% of the 40 test rebalances |
| H3 | ERC − GMV and ERC − MDP: Δα null | t within ±2 |
| H4 | ERC − EW < 0, driven by β | full-window t ≤ −2, a negative test-window mean, and the β part larger in size than Δα |
| H5 | ERC(LW) − ERC(S): null; ERC(S) − IV Δα: null | t within ±2 (full and test for LW) |

## §3 The math

**Risk contributions.** Portfolio volatility $\sigma_p = \sqrt{w'\Sigma w}$ is homogeneous of degree 1 in $w$, so Euler's theorem splits it exactly: $\sigma_p = \sum_i RC_i$ with $RC_i = w_i(\Sigma w)_i/\sigma_p$. Asset $i$'s variance share is $RC_i/\sigma_p = w_i\beta_{i,p}$, where $\beta_{i,p} = (\Sigma w)_i/(w'\Sigma w)$ is its beta to the portfolio.

**Risk budgeting (general form).** Given budgets $b_i > 0$ with $\sum_i b_i = 1$, find $w \ge 0$, $\mathbf 1'w = 1$ with $w_i\beta_{i,p} = b_i$ for every $i$. The long-only solution is unique and solves

$$\min_{y>0}\ \tfrac12\, y'\Sigma y - \sum_i b_i \log y_i, \qquad w = y/\mathbf 1'y .$$

The first-order condition $y_i(\Sigma y)_i = b_i$ delivers the budgets exactly. The log term keeps every weight strictly positive, so there is no boundary case and no fallback.

**ERC** sets $b_i = 1/N$; the class has equal budgets only. Two identities follow.

- Every variance share is $1/N$: $w_i\beta_{i,p} = 1/N$. With $\beta_{i,p} = \rho_{i,p}\sigma_i/\sigma_p$, $\;w_i = \dfrac{\sigma_p}{N\,\sigma_i\,\rho_{i,p}}$, so an asset gets a large capital weight if its volatility is low, its correlation with the portfolio is low, or both. Every $\beta_{i,p}$ is positive: no asset hedges an ERC portfolio.
- The risk weights $x = w\circ\sigma/(w'\sigma)$ are ERC on the correlation matrix $C = D^{-1}\Sigma D^{-1}$, so ERC's risk weights are scale-invariant, like MDP's in notebook 05.

**Special cases.** With equal correlations (any $N$), or with $N = 2$, ERC is inverse-vol: $w_i \propto 1/\sigma_i$.

**Contrast with GMV.** Long-only GMV's first-order condition gives $\beta_{i,p} = 1$ on every held asset: GMV equalizes marginal risk over the assets it holds, ERC equalizes variance shares over all 13. Volatility is ordered $\sigma_{GMV} \le \sigma_{ERC} \le \sigma_{EW}$.

## §4 The class and the rolling backtest

`EqualRiskContribution` estimates Σ with `_estimate` (defined in notebook 01 §3) and passes it to `_erc_long_only` in `src/maplab/models.py`. Each coordinate-descent sweep solves $\Sigma_{ii}y_i^2 + c_i y_i - 1/N = 0$ for its positive root, with $c_i = \sum_{j\ne i}\Sigma_{ij}y_j$, starting from inverse vol. It stops when every $y_i(\Sigma y)_i$ is within $10^{-12}$ of $1/N$ and raises after 1,000 sweeps. At all 208 rebalances, under the sample and Ledoit–Wolf Σ, every variance share equals 1/13, both §3 identities and the equal-correlation case hold, volatility is ordered $\sigma_{GMV} \le \sigma_{ERC} \le \sigma_{EW}$ and every weight is positive; `tests/test_nb06_checks.py` checks all of these.

### Backtests

Seven runs, harness as in notebook 01 §3. ERC(S) is registered, ERC(LW) is the sensitivity and IV the reference. IV is the package class `InverseVol`, with the same weights as the notebook-local IV in the registration. GMV(S) and MDP(S) are the runs of notebooks 01 and 05, for comparison; EW and 60/40 are the benchmarks. `tests/test_reproduction.py` checks that those four match the earlier notebooks.

In [ ]:
strategies = {
    "ERC(S)": EqualRiskContribution(cov_estimator=ml.sample_cov),
    "GMV(S)": GMV(cov_estimator=ml.sample_cov),
    "EW": EqualWeight(),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40"),
    "MDP(S)": MostDiversified(cov_estimator=ml.sample_cov),
    "ERC(LW)": EqualRiskContribution(cov_estimator=ml.ledoit_wolf_cov),
    "IV": ml.InverseVol(),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

In [ ]:
summary_table = cm.summary_by_window(results, split_ts, rf_daily, cost_drag=True)

## §5 Snapshots

### 5.1 Three dates

ERC, GMV, MDP, MaxSharpe and EW on the sample Σ at the three dates that fixed the H1/H2 directions: ex-ante volatility, effective N, and UUP's capital weight and variance share. MaxSharpe appears in snapshots only; it is not backtested.

In [ ]:
DATES = [pd.Timestamp(d) for d in ["2014-12-31", "2017-12-31", "2022-12-31"]]

snap_table, _, gmv_beta_dev = h.snapshot_tables(panel, DATES)

keep = ["GMV", "MDP", "ERC", "MaxSharpe", "EW"]
snap_shown = (snap_table.loc[snap_table.index.get_level_values("strategy").isin(keep),
                             ["exante_vol", "effN", "w_UUP", "UUP_rc_share"]]
              .rename(columns={"exante_vol": "ex-ante vol (%)", "effN": "effective N",
                               "w_UUP": "UUP weight (%)", "UUP_rc_share": "UUP variance share (%)"}))
display(ml.reporting.show_table(snap_shown, pct=["ex-ante vol (%)", "UUP weight (%)", "UUP variance share (%)"],
                                pct_dp=2, dp=2))

print(f"GMV: held assets' β to the portfolio = 1 to {max(gmv_beta_dev.values()):.1e} at all three dates (SLSQP tolerance)")

**Reading.** At all three dates ERC holds less UUP than GMV and MDP and has a higher effective N than both, the directions registered for H1 and H2. Under GMV every held asset has $\beta_{i,p} = 1$ (to 5.2e-05, SLSQP precision), so UUP's variance share equals its capital weight: 58.12% at 2022-12-31. Under ERC it is 7.69% (1/13).

### 5.2 Snapshot map at 2022-12-31

ERC, GMV, MDP, MaxSharpe, EW and 60/40 in (volatility, μ) space with the long-only frontier. μ is the 252-day sample mean, so read the x-axis.

In [ ]:
asof_2022 = pd.Timestamp("2022-12-31")
mu_2022, Sigma_2022 = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
Sigma_np = Sigma_2022.to_numpy()
n = Sigma_np.shape[0]
tickers = list(Sigma_2022.columns)

w_gmv_2022 = _min_variance_long_only(Sigma_2022)
w_mdp_2022, _ = _mdp_long_only(Sigma_2022, "snapshot", asof_2022)
w_erc_2022, _ = _erc_long_only(Sigma_2022, "snapshot", asof_2022)
rf_ann_2022 = MaxSharpe()._rf_ann(panel, asof_2022)
excess_2022 = mu_2022 - rf_ann_2022
w_msr_2022, _ = _tangency_long_only(excess_2022.to_numpy(), Sigma_np, "snapshot", asof_2022)
w_ew_2022 = np.full(n, 1.0 / n)
w_6040_2022 = FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40")(panel, asof_2022).reindex(tickers).to_numpy()

fig = ml.plotting.snapshot_map(
    Sigma_2022, mu_2022,
    {"ERC": w_erc_2022, "GMV": w_gmv_2022, "MDP": w_mdp_2022,
     "MaxSharpe": w_msr_2022, "EqualWeight": w_ew_2022, "60/40": w_6040_2022},
    title=f"Snapshot {asof_2022.date()}: ERC, GMV, MDP, MaxSharpe, EW, 60/40",
    figsize=(10, 6), legend_outside=True,
    styles={"60/40": dict(marker="D", s=130, color=ml.plotting.RUN_COLORS["60/40"])},
)
plt.show()

iU2022 = tickers.index("UUP")

**Reading.** ERC sits just right of GMV and MDP at the low-volatility end (ex-ante 4.32% against 3.32% and 3.50%), well left of EW (11.04%). Its UUP weight, 48.36%, is below GMV's 58.12%, MDP's 59.22% and MaxSharpe's 74.41%, and far above EW's 7.69%.

### 5.3 Capital and variance share at the three dates

Capital weight (dark) and variance share (light) per asset for GMV, MDP, ERC and EW; UUP in red. ERC's variance shares sit on the 1/13 line by construction; the gap between UUP's two bars is the corner.

In [ ]:
fig, _ = h.mechanism_grid(panel, DATES)
plt.show()

**Reading.** GMV's two bars match on every held asset ($\beta_{i,p} = 1$); MDP's do not. ERC's variance shares sit on 1/13 while UUP's capital bar stands at 25.74%, 24.14% and 48.36%. Under EW, UUP's variance share is negative at all three dates (−1.54%, −3.64%, −4.19%, §5.1 table): it hedges EW.

### 5.4 The ERC identity at 2022-12-31

Variance shares, betas to the portfolio and the weight formula $w_i = \sigma_p/(N\sigma_i\rho_{i,p})$ at the solved ERC weights.

In [ ]:
S_22 = Sigma_2022.to_numpy()
sig_22 = np.sqrt(np.diag(S_22))
var_22 = float(w_erc_2022 @ S_22 @ w_erc_2022)
sp_22 = np.sqrt(var_22)
beta_22 = (S_22 @ w_erc_2022) / var_22
rho_22 = (S_22 @ w_erc_2022) / (sig_22 * sp_22)
share_22 = w_erc_2022 * beta_22
w_formula_22 = sp_22 / (n * sig_22 * rho_22)
others = [i for i in range(n) if i != iU2022]
kkt = dict(
    vs_dev=float(np.max(np.abs(share_22 - 1.0 / n))),
    wf_dev=float(np.max(np.abs(w_formula_22 - w_erc_2022))),
    beta_min=float(beta_22.min()), beta_min_name=tickers[int(np.argmin(beta_22))],
    beta_uup=float(beta_22[iU2022]), rho_uup=float(rho_22[iU2022]),
    w_uup=float(w_erc_2022[iU2022]), nw_uup=float(n * w_erc_2022[iU2022]),
    vol_f=float(sp_22 / sig_22[iU2022]), vol_f_med=float(np.median(sp_22 / sig_22[others])),
    corr_f=float(1.0 / rho_22[iU2022]), corr_f_med=float(np.median(1.0 / rho_22[others])),
)
print(f"variance shares = 1/13 to {kkt['vs_dev']:.1e}; weights = σ_p/(Nσ_iρ_i,p) to {kkt['wf_dev']:.1e}; "
      f"smallest β to the portfolio: {kkt['beta_min_name']} {kkt['beta_min']:.4f}")
kkt_shown = pd.DataFrame(
    {"σ_p/σ_i": [kkt["vol_f"], kkt["vol_f_med"]], "1/ρ_i,p": [kkt["corr_f"], kkt["corr_f_med"]]},
    index=["UUP", "other 12 (median)"])
display(ml.reporting.show_table(kkt_shown, dp=3))

**Reading.** UUP's large weight comes mostly from its low correlation with the portfolio: against the median of the other 12, low volatility raises its weight about 1.9-fold and low correlation about 7-fold. Every variance share equals 1/13 and all 13 betas to the portfolio are positive, the smallest UUP's 0.1591. UUP's weight, 48.36%, is 6.29 × 1/13, and 6.29 = 0.466 ($\sigma_p/\sigma_i$) × 13.49 ($1/\rho_{i,p}$, with $\rho_{i,p}$ = 0.0741).

## §6 Out-of-sample results

### 6.1 Summary

Walk-forward as in notebook 01 §5; the test window (2023+) is the cleaner read, and wealth paths and formal tests are in notebook 08.

In [ ]:
cm.show_overview(cm.overview_table(summary_table, order=["ERC(S)", "ERC(LW)", "IV", "GMV(S)", "MDP(S)", "EW", "60/40"]))

**Reading.** ERC(S) runs at 3.94% volatility, between GMV(S)'s 3.17% and EW's 7.87% as §3 orders them, with turnover of 27%/yr (EW 14%, MDP(S) 52%). Its full-window Sharpe ratio is 0.76 (GMV(S) 0.62, EW 0.74, 60/40 0.91), and 0.94 in the test window against 60/40's 1.00.

### 6.2 Paired differences (H3–H5)

For the five registered pairs: daily return differences, each run's CAPM α and β, the regression of each difference on the market (Δα with its own t), and the β split total = β part + Δα; plain OLS, no HAC (notebook 08 uses Newey–West).

In [ ]:
pairs = [
    ("ERC(S)", "GMV(S)"),
    ("ERC(S)", "MDP(S)"),
    ("ERC(S)", "EW"),
    ("ERC(LW)", "ERC(S)"),
    ("ERC(S)", "IV"),
]

paired_table, paired_series = ml.inference.paired_table(results, pairs, split_ts, summary_table)
paired_shown = paired_table.loc[paired_table.index.get_level_values("window") != "train (<= split)",
                                 ["ann_mean_diff", "TE", "t", "cost_drag_diff"]].rename(columns={
    "ann_mean_diff": "mean diff (%/yr)", "TE": "TE (%/yr)", "cost_drag_diff": "cost-drag diff (%/yr)"})
ml.reporting.show_table(paired_shown, pct=["mean diff (%/yr)", "TE (%/yr)", "cost-drag diff (%/yr)"], pct_dp=2, dp=2)

**Reading.** Only ERC − GMV and ERC − EW cross $\lvert t \rvert$ = 2, and only over the full window. ERC(S) − GMV(S) is +1.04%/yr (t 2.47) full but t 1.32 test; ERC(S) − MDP(S) is +0.40%/yr (t 0.99). ERC(S) − EW is −2.78%/yr (t −2.08) and −3.16%/yr in the test window; ERC(LW) − ERC(S) has t 1.75 full and 1.24 test. Cost-drag differences are at most 0.05%/yr, so costs change no sign.

Single-strategy CAPM α and β, 60/40 included, are context only and not registered; notebook 08 tests each run's CAPM α with multiple-testing control.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_table = ml.inference.capm_table(results, ["ERC(S)", "GMV(S)", "MDP(S)", "EW", "60/40", "ERC(LW)", "IV"],
                                     MKT_simple, rf_daily)
capm_shown = capm_table.rename(columns={"alpha_ann": "α (%/yr)", "t_alpha": "t(α)", "beta": "β", "r2": "R²"}).rename_axis(None)
ml.reporting.show_table(capm_shown, pct=["α (%/yr)"], pct_dp=2, dp=2, int_cols=["n"])

In [ ]:
# Δα with its own t (the H3/H4/H5 statistic), not α_a − α_b; shown in the β-split table below
diff_capm_table = ml.inference.diff_regression(paired_series, pairs, MKT_simple)

In [ ]:
decomp = ml.inference.beta_decomposition(capm_table, paired_table, diff_capm_table, pairs,
                                         results, rf_daily, MKT_simple)
assert all(abs(r["identity_check"]) < 1e-9 for r in decomp.values()), "decomposition does not reproduce the difference"

decomp_shown = pd.DataFrame({
    pair: {"Δα (%/yr)": r["d_alpha"], "t(Δα)": r["t_alpha_diff"], "Δβ": r["d_beta"],
           "β part (%/yr)": r["beta_part"], "difference (%/yr)": r["realized_diff_full"]}
    for pair, r in decomp.items()}).T.astype(float)
pd.concat([ml.reporting.show_table(decomp_shown[["Δα (%/yr)", "t(Δα)"]], pct=["Δα (%/yr)"], pct_dp=2, dp=2),
           ml.reporting.show_table(decomp_shown[["Δβ"]], dp=4),
           ml.reporting.show_table(decomp_shown[["β part (%/yr)", "difference (%/yr)"]],
                                   pct=["β part (%/yr)", "difference (%/yr)"], pct_dp=2)], axis=1)

β split (full window): ERC(S) − GMV(S) is β part +0.95%/yr plus Δα +0.09%/yr (t 0.29), so the gap is ERC's higher β, which H3 sets aside by testing Δα. ERC(S) − EW's β part, −3.53%/yr, exceeds the whole −2.78%/yr shortfall, against Δα +0.76%/yr (t 0.89). ERC(S) − MDP(S) and ERC(S) − IV have Δα t −0.74 and 0.56.

Cumulative paired daily differences. Top: pairs registered on the total difference. Bottom: β-adjusted (solid, $d_t - \hat\beta\,\mathrm{mkt}_t$) and total (dashed) for the pairs registered on Δα.

In [ ]:
alpha_pairs = [("ERC(S)", "GMV(S)"), ("ERC(S)", "MDP(S)"), ("ERC(S)", "IV")]
total_pairs = [("ERC(S)", "EW"), ("ERC(LW)", "ERC(S)")]

fig, axes = plt.subplots(2, 1, figsize=(9.5, 9), sharex=True)
ml.plotting.cumulative_paired(
    paired_series, total_pairs, split_ts, ax=axes[0],
    title="(a) registered on the total difference (H4, H5 LW)", ylabel="cumulative diff (%)",
)
ml.plotting.cumulative_paired(
    paired_series, alpha_pairs, split_ts,
    beta={f"{a} - {b}": diff_capm_table.loc[f"{a} - {b}", "beta_diff"] for a, b in alpha_pairs},
    mkt_excess=MKT_simple, ax=axes[1],
    title="(b) registered on Δα (H3, H5 IV): solid = β-adjusted, dashed = total",
)
plt.tight_layout(); plt.show()

**Reading.** The endpoints match 17.2 years × the annualized means: in (a) ERC(S) − EW ends near −47.8% and ERC(LW) − ERC(S) near +3.2%; in (b) the totals against GMV, MDP and IV end near +17.9%, +6.8% and −17.1%, the β-adjusted lines near +1.6%, −4.5% and +5.6%. ERC − EW loses most in 2009–2012 and 2020–21, when equities rallied. ERC − MDP's β-adjusted line turns negative in 2022.

## §7 What drove the results

### 7.1 Weights by asset group

Capital by asset group over time for ERC(S), MDP(S) and GMV(S). Below is the mean over the 208 rebalances, with EW and 60/40 for reference (%).

In [ ]:
fig = ml.plotting.group_stackplot(results, ["ERC(S)", "MDP(S)", "GMV(S)"], split_ts=split_ts)
plt.show()

In [ ]:
group_means = {}
for name in ["ERC(S)", "MDP(S)", "GMV(S)", "EW", "60/40"]:
    g = ml.diagnostics.weights_by_group(results[name]["wlog"])
    assert len(g) == 208, (name, len(g))
    group_means[name] = g.mean()
ml.reporting.show_table((100 * pd.DataFrame(group_means)).rename(index=ml.plotting.GROUP_LABELS).rename_axis(None), dp=1)

**Reading.** ERC is less of a dollar portfolio than MDP or GMV: on average ERC(S) holds 32.0% in fx (UUP) and 10.1% in equity; MDP(S) holds 44.8% and 13.6%, GMV(S) 37.0% and 7.0%. ERC(S) holds the most credit (19.0%; MDP(S) 1.8%, GMV(S) 12.9%), and its inflation-linked weight, 9.7%, sits between MDP(S)'s 0.8% and GMV(S)'s 15.7%.

### 7.2 The UUP corner (H1)

UUP target weight under ERC(S) against GMV(S) and MDP(S) at the 208 rebalances (40 in test), with $\beta_{UUP,p}$ at 2022-12-31 as context.

In [ ]:
w_uup_erc_ts = results["ERC(S)"]["wlog"]["UUP"]
w_uup_gmv_ts = results["GMV(S)"]["wlog"]["UUP"]
w_uup_mdp_ts = results["MDP(S)"]["wlog"]["UUP"]

lt_gmv = w_uup_erc_ts < w_uup_gmv_ts
lt_mdp = w_uup_erc_ts < w_uup_mdp_ts
h1_full = lt_gmv & lt_mdp
n_full = len(h1_full)
test_mask = h1_full.index > split_ts
h1_test = h1_full.loc[test_mask]
n_test = int(test_mask.sum())

uup_2022 = {
    "ERC(S)": float(w_uup_erc_ts.loc[asof_2022]),
    "GMV(S)": float(w_uup_gmv_ts.loc[asof_2022]),
    "MDP(S)": float(w_uup_mdp_ts.loc[asof_2022]),
}

n_uup_top = int((results["ERC(S)"]["wlog"].idxmax(axis=1) == "UUP").sum())

print(f"H1 (ERC<GMV and ERC<MDP): full {int(h1_full.sum())}/{n_full} "
      f"({100*h1_full.sum()/n_full:.1f}%), test {int(h1_test.sum())}/{n_test} "
      f"({100*h1_test.sum()/n_test:.1f}%)")
print(f"2022-12-31 UUP weight: ERC={100 * uup_2022['ERC(S)']:.2f}%  GMV={100 * uup_2022['GMV(S)']:.2f}%  "
      f"MDP={100 * uup_2022['MDP(S)']:.2f}%")
print(f"2022-12-31 β of UUP to the ERC portfolio = {kkt['beta_uup']:.4f}")
print(f"UUP is ERC(S)'s largest weight at {n_uup_top} of {n_full} rebalances")

**Reading.** ERC(S) holds less UUP than both GMV(S) and MDP(S) at 197/208 rebalances (33/40 test) and at 2022-12-31: 48.36% against 58.12% and 59.22%. $\beta_{UUP,p}$ = 0.1591 is positive, so UUP diversifies the ERC portfolio rather than hedging it, and by §3 a low beta needs a large capital weight to carry 1/13 of the variance. UUP is ERC(S)'s largest weight at 207 of 208 rebalances (IEF is larger at 2016-06-30, 16.48% vs 16.17%) and averages 32.0%.

### 7.3 Concentration and turnover (H2)

Effective N of ERC(S) against the larger of GMV(S)'s and MDP(S)'s at the 208 rebalances (40 in test).

In [ ]:
effN_erc_ts = 1.0 / (results["ERC(S)"]["wlog"] ** 2).sum(axis=1)
effN_gmv_ts = 1.0 / (results["GMV(S)"]["wlog"] ** 2).sum(axis=1)
effN_mdp_ts = 1.0 / (results["MDP(S)"]["wlog"] ** 2).sum(axis=1)

comparator_max = pd.concat([effN_gmv_ts, effN_mdp_ts], axis=1).max(axis=1)
h2_full = effN_erc_ts >= comparator_max
n_full2 = len(h2_full)
test_mask2 = h2_full.index > split_ts
h2_test = h2_full.loc[test_mask2]
n_test2 = int(test_mask2.sum())

print(f"H2 (effN(ERC) >= max(GMV,MDP)): full {int(h2_full.sum())}/{n_full2} "
      f"({100*h2_full.sum()/n_full2:.1f}%), test {int(h2_test.sum())}/{n_test2} "
      f"({100*h2_test.sum()/n_test2:.1f}%)")
print(f"median effN: ERC(S)={float(effN_erc_ts.median()):.2f}  "
      f"GMV(S)={float(effN_gmv_ts.median()):.2f}  MDP(S)={float(effN_mdp_ts.median()):.2f}")
print(f"2022-12-31 effN: ERC={float(effN_erc_ts.loc[asof_2022]):.2f}  "
      f"GMV={float(effN_gmv_ts.loc[asof_2022]):.2f}  MDP={float(effN_mdp_ts.loc[asof_2022]):.2f}")

Effective N, UUP weight and trailing 12-rebalance one-way turnover (%/yr, initial allocation excluded); EW dashed, 60/40 as a reference line.

In [ ]:
fig = ml.plotting.concentration_panel(
    results, ["ERC(S)", "MDP(S)", "GMV(S)", "60/40"], rows=("effN", "highlight", "turnover"), ref="EW", split_ts=split_ts,
    figsize=(9.5, 10.5), colors={"60/40": ml.plotting.RUN_COLORS["60/40"]},
    suptitle="ERC(S) vs MDP(S) and GMV(S), EW and 60/40 for reference: effective N, UUP weight, turnover",
)
plt.show()

**Reading.** ERC(S)'s effective N is at least the larger of GMV(S)'s and MDP(S)'s at 208/208 rebalances (40/40 test); medians 6.86, 3.79 and 3.35, but only 3.88 at 2022-12-31 (GMV 2.51, MDP 2.47): equal variance shares are not equal capital. Its trailing turnover stays between about 18% and 39%/yr, below MDP(S)'s and GMV(S)'s throughout.

## Verdict table

The §2 rules applied to the statistics above: H1 in §7.2, H2 in §7.3, H3–H5 in §6.2. The rule column shows the registered wording; `[applied: …]` states how an ambiguous rule was read.

In [ ]:
reg = ml.inference.load_registration("nb06")

h1_holds_2022 = (uup_2022["ERC(S)"] < uup_2022["GMV(S)"]) and (uup_2022["ERC(S)"] < uup_2022["MDP(S)"])
d4 = decomp["ERC(S) - EW"]
t_full_ew = float(paired_table.loc[("ERC(S) - EW", "full"), "t"])
mean_test_ew = float(paired_table.loc[("ERC(S) - EW", "test (> split)"), "ann_mean_diff"])
t_full_lw = float(paired_table.loc[("ERC(LW) - ERC(S)", "full"), "t"])
t_test_lw = float(paired_table.loc[("ERC(LW) - ERC(S)", "test (> split)"), "t"])

stats = {
    "H1": dict(holds_2022=h1_holds_2022, frac_full=h1_full.sum() / n_full, frac_test=h1_test.sum() / n_test,
               display=f"full {int(h1_full.sum())}/{n_full}, test {int(h1_test.sum())}/{n_test}, "
                       f"2022 holds={h1_holds_2022}"),
    "H2": dict(frac_full=h2_full.sum() / n_full2, frac_test=h2_test.sum() / n_test2,
               display=f"full {int(h2_full.sum())}/{n_full2}, test {int(h2_test.sum())}/{n_test2}"),
}
for hid, pair_key in [("H3_GMV", "ERC(S) - GMV(S)"), ("H3_MDP", "ERC(S) - MDP(S)"), ("H5_IV", "ERC(S) - IV")]:
    t = float(diff_capm_table.loc[pair_key, "t_alpha_diff"])
    stats[hid] = dict(t=t, display=f"t={t:.2f}")
stats["H4"] = dict(t_full=t_full_ew, mean_test=mean_test_ew, beta_part=d4["beta_part"], d_alpha=d4["d_alpha"],
                   display=f"t_full={t_full_ew:.2f}, mean_test={100 * mean_test_ew:.2f}%, "
                           f"|β part|={100 * abs(d4['beta_part']):.2f}% vs |Δα|={100 * abs(d4['d_alpha']):.2f}%")
stats["H5_LW"] = dict(t_full=t_full_lw, t_test=t_test_lw, display=f"t_full={t_full_lw:.2f}, t_test={t_test_lw:.2f}")

vt = ml.inference.evaluate(reg, stats)
with pd.option_context("display.max_colwidth", None):
    display(vt.set_index("item"))

## §8 TAKEAWAY — equal risk shrinks the UUP corner but does not remove it

- **H1 supported: the UUP corner shrinks but survives.** ERC(S) holds less UUP than GMV(S) and MDP(S) at 197/208 rebalances (33/40 test); at 2022-12-31 the weights are 48.36% against 58.12% and 59.22%.
- **Why UUP still gets that much.** Every variance share is 1/13, so $w_i = \sigma_p/(N\sigma_i\rho_{i,p})$. UUP's low volatility and its low correlation with the portfolio ($\rho_{i,p}$ = 0.0741) both raise its weight; correlation by more. Its beta to the portfolio is positive: it diversifies, it does not hedge.
- **H2 supported.** Effective N is at least the larger of GMV's and MDP's at 208/208 rebalances (40/40 test). The median is 6.86 against 3.79 and 3.35, but only 3.88 at 2022-12-31.
- **Returns.** H3 nulls hold on Δα (t 0.29 and −0.74); ERC(S)'s gap over GMV(S) is mostly β. H4 supported: ERC(S) − EW is −2.78%/yr (t −2.08), β-driven. H5 nulls hold, LW narrowly (t 1.75).
- **Benchmarks (descriptive).** Full-window Sharpe ratio 0.76 against EW's 0.74 and 60/40's 0.91; tests are in notebook 08.

Equal risk shares fix UUP's share of variance at 1/13, not its share of capital.